# Camada Silver — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

---

## Visão Geral

A camada Silver é responsável por transformar os dados brutos ingeridos na Bronze em tabelas limpas, padronizadas e semanticamente organizadas segundo a **modelagem dimensional** (estrela). Cada notebook desta camada lê de `bronze.*`, aplica as transformações necessárias e grava em `silver.*`.

Os princípios aplicados em todas as tabelas Silver são:

- **Deduplicação** — reprocessamentos da Bronze (modo `append`) são neutralizados via `row_number()` sobre a chave primária ordenada por `timestamp_ingestion`
- **Tipagem explícita** — colunas lidas como `string` na Bronze recebem o tipo correto (`timestamp`, `decimal`, `boolean`, `int`)
- **Sanitização textual** — valores categóricos inconsistentes (variações de case, abreviações, typos) são normalizados para um conjunto canônico
- **Colunas derivadas** — atributos calculáveis a partir dos dados brutos são materializados para evitar recomputação nas camadas superiores
- **Idempotência** — todas as escritas usam `mode=overwrite`, garantindo que reexecutar o notebook produza sempre o mesmo resultado
- **Rastreabilidade** — `timestamp_ingestion` é recriado com o instante da carga Silver

---

## Tabelas produzidas neste notebook

### Tabelas Fato

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `ft_tickets_suporte` | `suporte_tickets` | Tickets do SAC limpos, tipados e com colunas derivadas |
| `ft_pedidos` | `pedidos` | Histórico de compras com valor total e ano_mes derivados |
| `ft_avaliacoes` | `avaliacoes` | Avaliações pós-compra com categoria NPS derivada |
| `ft_clickstream` | `clickstream` | Eventos de navegação tipados e normalizados |

### Dimensões

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `dim_clientes` | `clientes` | Perfis cadastrais com idade, nome completo e região derivados |
| `dim_produtos` | `catalogo_produtos` | Catálogo de produtos com categoria normalizada |
| `dim_tipos_problema` | `suporte_tickets` | Tipos de problema canônicos com categoria de negócio |
| `dim_agentes_suporte` | `suporte_tickets` | Agentes com métricas agregadas de desempenho |
| `dim_status_pedido` | `pedidos` | Status de pedido normalizados |
| `dim_categorias_produto` | `catalogo_produtos` | Categorias de produto normalizadas |

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalogo       = 'vcommerce_catalog'
bronze_schema  = 'vcommerce_bronze'
silver_schema  = 'vcommerce_silver'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {silver_schema}')
spark.sql(f'USE SCHEMA {silver_schema}')

print(f'Catálogo : {catalogo}')
print(f'Schema   : {silver_schema}')

---

## Tratamento: `ft_tickets_suporte`

**Origem:** `bronze.suporte_tickets`  
**Destino:** `silver.ft_tickets_suporte`, `silver.dim_tipos_problema`, `silver.dim_agentes_suporte`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_problema` | ~25 variações para 4 categorias reais (`pro`, `p3oduto`, `PRODUCT`, `DELAY`…) | Mapeamento para valores canônicos |
| `data_resolucao` | 2.072 nulos — tickets ainda abertos | Mantidos como `null`; `resolvido = false` |
| `nota_avaliacao` | 2.072 nulos — sem avaliação para tickets abertos | Mantidos como `null` |
| `tempo_resolucao_horas` | 2.072 nulos — tickets não resolvidos | Mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `ticket_id` |
| Datas | Formato ISO com timezone (`2023-01-03T05:13:00.000Z`) | Cast para `timestamp` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `resolvido` | `data_resolucao IS NOT NULL` |
| `hora_abertura` | `HOUR(data_abertura)` |
| `dia_semana_abertura` | `DAYOFWEEK(data_abertura)` |

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada ticket (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.suporte_tickets')

window_dedup = Window.partitionBy('ticket_id').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [0]:
# ─── Mapeamento canônico de tipo_problema ────────────────────────────────────
# Na Bronze foram encontradas 4 categorias reais fragmentadas em ~25 variações:
#
#   Entrega   - Entrega, ENTREGA, entrega, 3ntrega, entr, del, DELAY, delay
#   Reembolso - Reembolso, REEMBOLSO, reembolso, REFUND, ref, reemb, r3embolso
#   Produto   - Produto, PRODUTO, produto, pro, prod, p3oduto, PRODUCT, product
#   Pagamento - Pagamento, PAGAMENTO, pagamento, pag, pay, PAY, PAYMENT, payment,
#               p4gamento
#
# Estratégia: normaliza para lowercase e aplica mapeamento por prefixo/palavra-chave.
# Valores não mapeados são marcados como 'Outro' para investigação futura.

tipo_map = {
    # Entrega
    'entrega'  : 'Entrega',  '3ntrega' : 'Entrega',  'entr'  : 'Entrega',
    'del'      : 'Entrega',  'delay'   : 'Entrega',
    # Reembolso
    'reembolso': 'Reembolso','reemb'   : 'Reembolso','r3embolso': 'Reembolso',
    'refund'   : 'Reembolso','ref'     : 'Reembolso',
    # Produto
    'produto'  : 'Produto',  'pro'     : 'Produto',  'prod'  : 'Produto',
    'p3oduto'  : 'Produto',  'product' : 'Produto',
    # Pagamento
    'pagamento': 'Pagamento','pag'     : 'Pagamento','p4gamento': 'Pagamento',
    'pay'      : 'Pagamento','payment' : 'Pagamento',
}

# Constrói expressão CASE WHEN a partir do dicionário
tipo_expr = F.col('tipo_problema')
for raw_val, canonical in tipo_map.items():
    tipo_expr = F.when(
        F.lower(F.col('tipo_problema')) == raw_val, canonical
    ).otherwise(tipo_expr)

# Valores que não bateram com nenhum mapeamento ficam como 'Outro'
known_lower = [k.lower() for k in tipo_map.keys()]
tipo_expr = F.when(
    F.lower(F.col('tipo_problema')).isin(known_lower), tipo_expr
).otherwise(F.lit('Outro'))

print('Expressão de mapeamento criada.')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver = (
    df_dedup

    # 1. Tipos corretos para datas (ISO 8601 com timezone)
    .withColumn('data_abertura',   F.to_timestamp('data_abertura'))
    .withColumn('data_resolucao',  F.to_timestamp('data_resolucao'))

    # 2. Normalização de tipo_problema
    .withColumn('tipo_problema', tipo_expr)

    # 3. Colunas derivadas de negócio
    .withColumn(
        'resolvido',
        F.col('data_resolucao').isNotNull()
    )
    .withColumn(
        'hora_abertura',
        F.hour('data_abertura').cast('int')
    )
    .withColumn(
        'dia_semana_abertura',
        F.date_format('data_abertura', 'EEEE')   # nome completo em inglês; adaptar locale se necessário
    )

    # 4. Tempo resolução em minutos (substitui tempo_resolucao_horas)
    .withColumn(
        'tempo_resolucao_minutos',
        F.when(
            F.col('data_resolucao').isNotNull(),
            (F.col('data_resolucao').cast('long') - F.col('data_abertura').cast('long')) / 60
        ).otherwise(None)
    )

    # 5. Tipos numéricos
    .withColumn('nota_avaliacao',        F.col('nota_avaliacao').cast('decimal(3,1)'))

    # 6. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 7. Ordenação de colunas conforme schema Silver acordado
    .select(
        'ticket_id',
        'id_cliente',
        'id_pedido',
        'tipo_problema',
        'data_abertura',
        'data_resolucao',
        'tempo_resolucao_minutos',
        'agente_suporte',
        'nota_avaliacao',
        'resolvido',
        'hora_abertura',
        'dia_semana_abertura',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total = df_silver.count()
resolvidos     = df_silver.filter(F.col('resolvido') == True).count()
nao_resolvidos = df_silver.filter(F.col('resolvido') == False).count()
outros_tipo    = df_silver.filter(F.col('tipo_problema') == 'Outro').count()

print(f'Total de tickets       : {total:,}')
print(f'Resolvidos             : {resolvidos:,} ({resolvidos/total*100:.1f}%)')
print(f'Abertos (sem resolução): {nao_resolvidos:,} ({nao_resolvidos/total*100:.1f}%)')
print(f'Tipo "Outro" (suspeitos): {outros_tipo:,}')
print()
print('Distribuição de tipo_problema após normalização:')
df_silver.groupBy('tipo_problema').count().orderBy(F.desc('count')).show()

In [0]:
# ─── Grava silver.ft_tickets_suporte ─────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.ft_tickets_suporte')
)

print(f'{silver_schema}.ft_tickets_suporte gravada com {df_silver.count():,} registros.')

In [0]:
# ─── dim_tipos_problema ───────────────────────────────────────────────────────
# Dimensão com os 4 tipos canônicos e sua categoria de negócio.
#
# Categoria derivada:
#   Entrega   - Logística
#   Reembolso - Financeiro
#   Produto   - Qualidade
#   Pagamento - Financeiro
#   Outro     - Indefinido

df_dim_tipos = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .select('tipo_problema')
         .distinct()
         .withColumn(
             'categoria_problema',
             F.when(F.col('tipo_problema') == 'Entrega',   'Logística')
              .when(F.col('tipo_problema') == 'Reembolso', 'Financeiro')
              .when(F.col('tipo_problema') == 'Produto',   'Qualidade')
              .when(F.col('tipo_problema') == 'Pagamento', 'Financeiro')
              .otherwise('Indefinido')
         )
         .orderBy('tipo_problema')
)

df_dim_tipos.show()

(
    df_dim_tipos.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(f'{silver_schema}.dim_tipos_problema')
)

print(f'{silver_schema}.dim_tipos_problema gravada.')

In [0]:
# ─── dim_agentes_suporte ──────────────────────────────────────────────────────
# Dimensão com métricas agregadas por agente, derivadas de ft_tickets_suporte.
#
# Métricas calculadas:
#   qtd_tickets_resolvidos  - total de tickets onde resolvido = true
#   nota_media_atendimento  - média de nota_avaliacao (ignora nulos)

df_dim_agentes = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .groupBy('agente_suporte')
         .agg(
             F.count(
                 F.when(F.col('resolvido') == True, 1)
             ).alias('qtd_tickets_resolvidos'),
             F.round(
                 F.avg('nota_avaliacao'), 2
             ).alias('nota_media_atendimento'),
         )
         .withColumn(
             'nota_media_atendimento',
             F.col('nota_media_atendimento').cast('decimal(4,2)')
         )
         .orderBy(F.desc('qtd_tickets_resolvidos'))
)

df_dim_agentes.show(truncate=False)

(
    df_dim_agentes.write
                  .format('delta')
                  .mode('overwrite')
                  .option('overwriteSchema', 'true')
                  .saveAsTable(f'{silver_schema}.dim_agentes_suporte')
)

print(f'{silver_schema}.dim_agentes_suporte gravada.')

In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────

tabelas = [
    f'{silver_schema}.ft_tickets_suporte',
    f'{silver_schema}.dim_tipos_problema',
    f'{silver_schema}.dim_agentes_suporte',
]

print('=== Camada Silver — Tickets de Suporte ===')
print(f'{"Tabela":<35} {"Linhas":>8} {"Colunas":>8}')
print('-' * 55)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<35} {df_t.count():>8,} {len(df_t.columns):>8}')

---

## Tratamento: `ft_avaliacoes`

**Origem:** `bronze.avaliacoes`  
**Destino:** `silver.ft_avaliacoes`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `nota_produto` | Valores fora do range 1–5 (`-1`, `0`, `6`) e textos (`bom`, `ruim`, `péssimo`, `ótimo`) | Mapear textos para numérico; anular valores fora de 1–5 com `null` |
| `nota_nps` | Valores fora do range 0–10 (`-1`, `11`) e mesmos textos acima | Mapear textos para numérico; anular valores fora de 0–10 com `null` |
| `recomenda` | ~12 variações (`S`, `sim`, `SIM`, `yes`, `1` / `N`, `Nao`, `NAO`, `no`, `0`) | Normalizar para booleano (`true` / `false`) |
| `data_avaliacao` | 3.174 registros nulos; formato `datetime com espaço` | Cast para `timestamp`; nulos mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `id_avaliacao` |

### Colunas derivadas criadas

| Coluna | Lógica |
|---|---|
| `categoria_nps` | Detrator (0–6) · Neutro (7–8) · Promotor (9–10) — `null` quando `nota_nps` é inválida |
| `recomenda` | Convertido de texto/número para boolean |


In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Lê todos os registros brutos da tabela de avaliações ingerida na camada Bronze.
# Como a Bronze usa mode=append, reprocessamentos podem gerar registros duplicados
# para o mesmo id_avaliacao. Usamos row_number() sobre a janela particionada por
# id_avaliacao (ordenada pelo timestamp_ingestion mais recente) para ficar apenas
# com a versão mais atual de cada avaliação.

df_raw = spark.table(f'{bronze_schema}.avaliacoes')

window_dedup = Window.partitionBy('id_avaliacao').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

total_raw = df_raw.count()
total_dedup = df_dedup.count()

print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')


In [0]:
# ─── Mapeamento de notas textuais -> numéricas ────────────────────────────────
# Na Bronze, tanto nota_produto quanto nota_nps apresentam valores textuais
# misturados com valores numéricos. Os textos encontrados e seus equivalentes
# numéricos adotados são:
#
#   'ótimo'   -> 5   (nota máxima)
#   'bom'     -> 4   (nota boa)
#   'ruim'    -> 2   (nota ruim)
#   'péssimo' -> 1   (nota mínima)
#
# Após a conversão textual, valores fora dos ranges válidos são anulados:
#   nota_produto : range esperado 1–5  → fora disso vira null
#   nota_nps     : range esperado 0–10 → fora disso vira null

texto_para_nota = {
    'ótimo':   5,
    'bom':     4,
    'ruim':    2,
    'péssimo': 1,
}
# Constrói expressão que converte texto para número antes do cast final.
# Para cada par (texto, valor) no dicionário, encadeia um WHEN; o OTHERWISE
# mantém o valor original (caso já seja numérico em string).

def texto_para_numero_da_nota(coluna: str):
    # Essa função retorna uma coluna Pyspark que normaliza os textos para numero int
    expressao = F.col(coluna)
    for texto, nota in texto_para_nota.items():
        # Cast final para int e valores qque nao sao numericos que sobrarem viram null
        expressao = F.when(F.lower(F.col(coluna)) == texto, F.lit(nota)).otherwise(expressao)
    return expressao.cast('int')

print('Expressões de mapeamento criadas.')
print(f'Textos mapeados: {list(texto_para_nota.keys())}')

In [0]:
# ─── Mapeamento de recomenda -> boolean ───────────────────────────────────────
# O campo recomenda chegou com ~12 variações para dois valores semânticos:
#
#   Positivo (true)  : 'S', 'Sim', 'SIM', 'sim', 'yes', '1'
#   Negativo (false) : 'N', 'Nao', 'NAO', 'nao', 'no',  '0'
#
# A estratégia é normalizar para lowercase e checar pertencimento ao conjunto
# positivo. Qualquer valor não reconhecido vira null para não inferir intenção.

valor_recomenda_positivo = {'s', 'sim', 'yes', '1'}
valor_recomenda_negativo = {'n', 'nao', 'no', '0'}

expressao_recomenda = (
    F.when(F.lower(F.col('recomenda')).isin(valor_recomenda_positivo), True)
    .when(F.lower(F.col('recomenda')).isin(valor_recomenda_negativo), False)
    .otherwise(None) #aqui é para os valores que a gente nao conhece ai não inferimos
    .cast('boolean')
)

print('Expressão de normalização de recomenda criada.')
print(f'  Positivo (true) : {valor_recomenda_positivo}')
print(f'  Negativo (false): {valor_recomenda_negativo}')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────
# Aplica sequencialmente todos os tratamentos definidos acima sobre o DataFrame
# deduplicado, produzindo o DataFrame Silver final.

df_silver = (
    df_dedup

    # 1. Converte notas textuais para inteiro e anula valores fora do range válido
    #    nota_produto: range 1–5 (escala de satisfação do produto)
    #    nota_nps: range 0–10 (Net Promoter Score)
    .withColumn('nota_produto',
        F.when(
            texto_para_numero_da_nota('nota_produto').between(1, 5),
            texto_para_numero_da_nota('nota_produto')
        ).otherwise(F.lit(None).cast('int'))   # fora do range -> null
    )
    .withColumn('nota_nps',
        F.when(
            texto_para_numero_da_nota('nota_nps').between(0, 10),
            texto_para_numero_da_nota('nota_nps')
        ).otherwise(F.lit(None).cast('int'))   # fora do range -> null
    )

    # 2. Normaliza recomenda para boolean usando a expressão construída acima
    .withColumn('recomenda', expressao_recomenda)

     # 3. Cast de data_avaliacao para timestamp
    #    Dois formatos coexistem na Bronze:
    #      - Padrão  : 'yyyy-MM-dd HH:mm:ss'  -> maioria dos registros
    #      - Variante: 'dd/MM/yyyy HH:mm:ss'  -> ~12.5k registros com barras
    #    try_to_timestamp é usado no lugar de to_timestamp pois nunca lança
    #    exceção — retorna null quando o valor não bate com o formato, o que
    #    torna o coalesce seguro mesmo com dados malformados.
    .withColumn('data_avaliacao', 
        F.coalesce(
            # Tenta o formato padrão com hífen (Ano-Mês-Dia)
            F.expr("try_to_timestamp(data_avaliacao, 'yyyy-MM-dd HH:mm:ss')"),    
            # Tenta o formato com barras (Ano/Dia/Mês) - O primeiro que deu erro
            F.expr("try_to_timestamp(data_avaliacao, 'yyyy/dd/MM HH:mm:ss')"),
            # Tenta o formato Brasileiro com barras (Dia/Mês/Ano) - Visto no print
            F.expr("try_to_timestamp(data_avaliacao, 'dd/MM/yyyy HH:mm:ss')"), 
            # Tenta o formato Americano com hifens (Mês-Dia-Ano) - Visto no print
            F.expr("try_to_timestamp(data_avaliacao, 'MM-dd-yyyy HH:mm:ss')"),
            # Fallback nativo do Spark para tentar salvar o que sobrar
            F.expr("try_to_timestamp(data_avaliacao)")
        )
    )

    # 4. Categoria NPS derivada — classificação padrão de mercado:
    #      0–6  -> Detrator  (clientes insatisfeitos, risco de churn)
    #      7–8  -> Neutro    (clientes passivos, sem engajamento forte)
    #      9–10 -> Promotor  (clientes leais, propensos a indicar)
    #    Quando nota_nps é null (valor inválido na origem), categoria_nps
    #    também fica null para não distorcer análises de NPS.
    .withColumn('categoria_nps',
        F.when(F.col('nota_nps').between(0, 6),  'Detrator')
         .when(F.col('nota_nps').between(7, 8),  'Neutro')
         .when(F.col('nota_nps').between(9, 10), 'Promotor')
         .otherwise(None)
    )

    # 5. Marca temporal desta carga Silver (recriado para rastreabilidade)
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'id_avaliacao',
        'id_pedido',
        'id_cliente',
        'id_produto',
        'nota_produto',
        'comentario',
        'nota_nps',
        'categoria_nps',
        'recomenda',
        'data_avaliacao',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────
# Antes de gravar, verificamos as principais métricas de qualidade para garantir
# que os tratamentos produziram o resultado esperado. Qualquer número suspeito
# deve ser investigado antes de prosseguir.

total = df_silver.count()

# Notas que viraram null após tratamento (eram inválidas na origem)
notas_produto_nulas = df_silver.filter(F.col('nota_produto').isNull()).count()
notas_nps_nulas = df_silver.filter(F.col('nota_nps').isNull()).count()
recomenda_nulas = df_silver.filter(F.col('recomenda').isNull()).count()
datas_nulas = df_silver.filter(F.col('data_avaliacao').isNull()).count()

print(f'Total de avaliações: {total:,}')
print()
print(f'nota_produto nulas (inválidas): {notas_produto_nulas:,} ({notas_produto_nulas/total*100:.1f}%)')
print(f'nota_nps nulas (inválidas): {notas_nps_nulas:,} ({notas_nps_nulas/total*100:.1f}%)')
print(f'recomenda nulas: {recomenda_nulas:,} ({recomenda_nulas/total*100:.1f}%)')
print(f'data_avaliacao nulas: {datas_nulas:,} ({datas_nulas/total*100:.1f}%)')
print()
print('Distribuição de categoria_nps:')
df_silver.groupBy('categoria_nps').count().orderBy('categoria_nps').show()
print('Distribuição de recomenda:')
df_silver.groupBy('recomenda').count().orderBy('recomenda').show()
print('Distribuição de nota_produto (range 1–5):')
df_silver.groupBy('nota_produto').count().orderBy('nota_produto').show()

In [0]:
# ─── Grava silver.ft_avaliacoes ──────────────────────────────────────────────
# Usa mode=overwrite para garantir idempotência: reexecutar este notebook
# sempre produz o mesmo resultado, sem acumular registros duplicados.
# overwriteSchema=true permite que alterações de schema futuras não quebrem
# a execução.

df_silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{silver_schema}.ft_avaliacoes')

print(f'{silver_schema}.ft_avaliacoes gravadas com {df_silver.count():,} registros.')

In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────
# Confirma a tabela gravada e exibe contagem e número de colunas,
# seguindo o mesmo padrão de log das outras seções deste notebook.

tabelas = [
    f'{silver_schema}.ft_avaliacoes',
]

print('=== Camada Silver — Avaliações Pós-Compra ===')
print(f'{"Tabela":<40} {"Linhas":>8} {"Colunas":>8}')
print('-' * 60)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<40} {df_t.count():>8,} {len(df_t.columns):>8}')